In [42]:
# ================================================================
# 🍎 AI NUTRITION COACH
# Multimodal Food Recognition + Nutrition Estimation
# + Deterministic Calorie Calculation + Reliability Audit
#
# Model: HuggingFaceTB/SmolVLM2-2.2B-Instruct
# Hardware: Google Colab GPU (Tesla T4 or better recommended)
#
# NOTE:
# This project performs REFERENCE-FREE reliability evaluation.
# The reported metrics do NOT represent nutritional accuracy.
# ================================================================

# =========================
# 1. INSTALL DEPENDENCIES
# =========================

!pip -q install -U transformers accelerate torch torchvision pillow pandas numpy

# =========================
# 2. IMPORTS & CONFIGURATION
# =========================

import os
import re
import json
import time
import glob
import warnings
import numpy as np
import pandas as pd
import torch

from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

warnings.filterwarnings("ignore")

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

IMAGE_EXTENSIONS = [
    "*.jpg", "*.jpeg", "*.png",
    "*.webp", "*.avif"
]

OUTPUT_CSV = "/content/AI_Nutrition_Final_Validation.csv"

# =========================
# 3. ENVIRONMENT
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 72)
print("🍎 AI NUTRITION COACH")
print("Multimodal AI Nutrition Analysis System")
print("=" * 72)

print(f"Python        : {os.sys.version.split()[0]}")
print(f"PyTorch       : {torch.__version__}")
print(f"Device        : {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"GPU           : {gpu_name}")
    print(f"GPU Memory    : {gpu_memory:.2f} GB")
else:
    print("GPU           : Not available")
    print("⚠️ CPU inference will be significantly slower.")

# =========================
# 4. LOAD MULTIMODAL MODEL
# =========================

print("\n" + "-" * 72)
print("Loading multimodal vision-language model...")
print("-" * 72)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

model.eval()

print("✓ Model loaded successfully")
print(f"✓ Model: {MODEL_ID}")

# =========================
# 5. FIND TEST IMAGES
# =========================

image_paths = []

for ext in IMAGE_EXTENSIONS:
    image_paths.extend(
        glob.glob(f"/content/{ext}")
    )

# Remove duplicate paths and sort
image_paths = sorted(set(image_paths))

# Prefer the known 12-image benchmark if present
known_keywords = [
    "amirali-mirhashemian",
    "Delicious-Cheeseburger",
    "images (1)",
    "images (2)",
    "images (3)",
    "images (6)",
    "istockphoto-471674664",
    "penne-pasta",
    "roasted-chicken",
    "shutterstock_16650619",
    "side-view-shawarma",
    "Single-Food-Salad"
]

preferred = []

for path in image_paths:
    name = os.path.basename(path)

    if any(k.lower() in name.lower() for k in known_keywords):
        preferred.append(path)

if len(preferred) >= 12:
    image_paths = sorted(preferred)[:12]

print("\n" + "-" * 72)
print("Evaluation Dataset")
print("-" * 72)

print(f"Images found : {len(image_paths)}")

if len(image_paths) == 0:
    raise FileNotFoundError(
        "No images found in /content. "
        "Upload your food images to Colab and rerun this cell."
    )

if len(image_paths) != 12:
    print(
        f"⚠️ Expected the 12-image benchmark, "
        f"but found {len(image_paths)} images."
    )

# =========================
# 6. MULTIMODAL INFERENCE
# =========================

def extract_json(text):
    """
    Extract the first valid JSON object from model output.
    """

    text = text.strip()

    # Remove markdown fences
    text = re.sub(
        r"```json|```",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    # Direct JSON
    try:
        return json.loads(text)
    except:
        pass

    # Search for JSON object
    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if match:
        candidate = match.group(0)

        try:
            return json.loads(candidate)
        except:
            pass

    return None


def safe_float(value):
    try:
        return float(value)
    except:
        return np.nan


def analyze_food(image_path):

    image = Image.open(image_path).convert("RGB")

    prompt = """
You are a conservative multimodal nutrition estimation assistant.

Analyze ONLY the food visibly present in the image.

Identify:
1. The visible food
2. A realistic approximate visible portion
3. Estimated protein in grams
4. Estimated carbohydrates in grams
5. Estimated fat in grams
6. Estimated fiber in grams
7. Confidence level
8. Main uncertainty

Rules:
- Do not invent brands unless clearly visible.
- Do not assume an unusually small portion.
- Use realistic approximate portions.
- If the exact portion cannot be determined, use an approximate portion.
- Do not output zero protein, carbohydrates, and fat when the visible food
  clearly contains meaningful macronutrients.
- Be conservative when visual evidence is uncertain.
- Use medium or low confidence when the portion or food identity is uncertain.
- Do NOT calculate calories.
- Return ONLY valid JSON.

Required JSON format:

{
  "food": "string",
  "portion": "string",
  "protein_g": 0.0,
  "carbs_g": 0.0,
  "fat_g": 0.0,
  "fiber_g": 0.0,
  "confidence": "low",
  "uncertainty": "string"
}
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    start = time.time()

    try:

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model.device)
            if hasattr(v, "to")
            else v
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generated_ids = model.generate(
                **inputs,
                max_new_tokens=220,
                do_sample=False
            )

        generated_text = processor.batch_decode(
            generated_ids[:, inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )[0]

        latency = time.time() - start

        result = extract_json(generated_text)

        if result is None:

            return {
                "food": "",
                "portion": "",
                "protein_g": np.nan,
                "carbs_g": np.nan,
                "fat_g": np.nan,
                "fiber_g": np.nan,
                "confidence": "low",
                "uncertainty": "JSON parsing failed",
                "json_valid": False,
                "latency_sec": latency
            }

        # Normalize fields
        food = str(result.get("food", "")).strip()
        portion = str(result.get("portion", "")).strip()

        protein = safe_float(result.get("protein_g"))
        carbs = safe_float(result.get("carbs_g"))
        fat = safe_float(result.get("fat_g"))
        fiber = safe_float(result.get("fiber_g"))

        confidence = str(
            result.get("confidence", "medium")
        ).lower().strip()

        if confidence not in ["low", "medium", "high"]:
            confidence = "medium"

        uncertainty = str(
            result.get("uncertainty", "")
        ).strip()

        # Deterministic calorie calculation
        if all(
            not pd.isna(x)
            for x in [protein, carbs, fat]
        ):

            calories = (
                4 * protein +
                4 * carbs +
                9 * fat
            )

        else:
            calories = np.nan

        return {
            "food": food,
            "portion": portion,
            "protein_g": protein,
            "carbs_g": carbs,
            "fat_g": fat,
            "fiber_g": fiber,
            "calculated_calories": calories,
            "confidence": confidence,
            "uncertainty": uncertainty,
            "json_valid": True,
            "latency_sec": latency
        }

    except Exception as e:

        latency = time.time() - start

        return {
            "food": "",
            "portion": "",
            "protein_g": np.nan,
            "carbs_g": np.nan,
            "fat_g": np.nan,
            "fiber_g": np.nan,
            "calculated_calories": np.nan,
            "confidence": "low",
            "uncertainty": str(e),
            "json_valid": False,
            "latency_sec": latency
        }


# =========================
# 7. RUN BENCHMARK
# =========================

print("\n" + "-" * 72)
print("Running Multimodal Food Analysis")
print("-" * 72)

results = []

for i, path in enumerate(image_paths, start=1):

    filename = os.path.basename(path)

    print(
        f"[{i:02d}/{len(image_paths)}] "
        f"{filename[:55]}"
    )

    result = analyze_food(path)

    result["image"] = filename

    results.append(result)

df = pd.DataFrame(results)

# =========================
# 8. STRUCTURAL VALIDATION
# =========================

required_fields = [
    "food",
    "portion",
    "protein_g",
    "carbs_g",
    "fat_g",
    "fiber_g",
    "calculated_calories"
]

def complete_output(row):

    for field in required_fields:

        value = row.get(field)

        if pd.isna(value):
            return False

        if isinstance(value, str) and not value.strip():
            return False

    return True


df["complete_output"] = df.apply(
    complete_output,
    axis=1
)

numeric_fields = [
    "protein_g",
    "carbs_g",
    "fat_g",
    "fiber_g",
    "calculated_calories"
]

df["numeric_valid"] = df[numeric_fields].apply(
    lambda row: all(
        pd.notna(x) and x >= 0
        for x in row
    ),
    axis=1
)

# =========================
# 9. CALORIE CONSISTENCY
# =========================

df["macro_derived_calories"] = (
    4 * df["protein_g"] +
    4 * df["carbs_g"] +
    9 * df["fat_g"]
)

df["calorie_difference"] = (
    df["calculated_calories"] -
    df["macro_derived_calories"]
).abs()

df["macro_calorie_consistent"] = (
    df["calorie_difference"] < 0.01
)

# =========================
# 10. REFERENCE-FREE RISK AUDIT
# =========================

generic_labels = [
    "food",
    "dish",
    "meal",
    "item",
    "something",
    "unknown",
    "wrapped food",
    "food item",
    "mixed food"
]

def normalize_text(value):

    if pd.isna(value):
        return ""

    return str(value).strip().lower()


def zero_macro(row):

    return (
        row["protein_g"] == 0 and
        row["carbs_g"] == 0 and
        row["fat_g"] == 0
    )


def generic_food(row):

    food = normalize_text(row["food"])

    return (
        food == "" or
        food in generic_labels
    )


def very_low_calorie(row):

    calories = row["calculated_calories"]

    if pd.isna(calories):
        return True

    return (
        calories > 0 and
        calories < 50
    )


def low_macro_mass(row):

    values = [
        row["protein_g"],
        row["carbs_g"],
        row["fat_g"]
    ]

    if any(pd.isna(x) for x in values):
        return True

    return sum(values) < 3


df["zero_major_macros"] = df.apply(
    zero_macro,
    axis=1
)

df["generic_food_label"] = df.apply(
    generic_food,
    axis=1
)

df["very_low_calorie"] = df.apply(
    very_low_calorie,
    axis=1
)

df["low_macro_mass"] = df.apply(
    low_macro_mass,
    axis=1
)

# =========================
# 11. RISK SCORE
# =========================

def risk_score(row):

    score = 0

    if row["zero_major_macros"]:
        score += 4

    if row["generic_food_label"]:
        score += 3

    if row["very_low_calorie"]:
        score += 2

    if row["low_macro_mass"]:
        score += 2

    return score


df["risk_score"] = df.apply(
    risk_score,
    axis=1
)

# =========================
# 12. CONFIDENCE CALIBRATION
# =========================

def calibrate_confidence(row):

    original = normalize_text(row["confidence"])
    risk = row["risk_score"]

    if risk >= 5:
        return "low"

    if risk >= 2:
        return "medium"

    if original in ["low", "medium", "high"]:
        return original

    return "medium"


df["calibrated_confidence"] = df.apply(
    calibrate_confidence,
    axis=1
)

# =========================
# 13. REVIEW FLAG
# =========================

df["review_required"] = (
    df["risk_score"] >= 2
)

# =========================
# 14. DIAGNOSTIC REASONS
# =========================

def diagnostic_reason(row):

    reasons = []

    if row["zero_major_macros"]:
        reasons.append("zero_major_macros")

    if row["generic_food_label"]:
        reasons.append("generic_food_label")

    if row["very_low_calorie"]:
        reasons.append("very_low_calorie")

    if row["low_macro_mass"]:
        reasons.append("low_macro_mass")

    return "; ".join(reasons) if reasons else "none"


df["diagnostic_reason"] = df.apply(
    diagnostic_reason,
    axis=1
)

# =========================
# 15. FINAL QUALITY STATUS
# =========================

def quality_status(row):

    risk = row["risk_score"]

    if not row["json_valid"]:
        return "FAIL"

    if not row["numeric_valid"]:
        return "FAIL"

    if risk >= 5:
        return "FAIL"

    if risk >= 2:
        return "WARN"

    return "PASS"


df["quality_status"] = df.apply(
    quality_status,
    axis=1
)

# =========================
# 16. CONFIDENCE AUDIT
# =========================

df["original_overconfidence"] = (
    (df["confidence"] == "high") &
    (df["risk_score"] >= 2)
)

df["remaining_high_confidence_risk"] = (
    (df["calibrated_confidence"] == "high") &
    (df["risk_score"] >= 2)
)

# =========================
# 17. SYSTEM METRICS
# =========================

n = len(df)

json_validity = (
    df["json_valid"].mean() * 100
)

structured_completeness = (
    df["complete_output"].mean() * 100
)

numeric_validity = (
    df["numeric_valid"].mean() * 100
)

macro_consistency = (
    df["macro_calorie_consistent"].mean() * 100
)

zero_macro_rate = (
    df["zero_major_macros"].mean() * 100
)

generic_rate = (
    df["generic_food_label"].mean() * 100
)

low_calorie_rate = (
    df["very_low_calorie"].mean() * 100
)

original_overconfidence = (
    df["original_overconfidence"].mean() * 100
)

remaining_overconfidence = (
    df["remaining_high_confidence_risk"].mean() * 100
)

review_rate = (
    df["review_required"].mean() * 100
)

pass_rate = (
    (df["quality_status"] == "PASS").mean() * 100
)

warn_rate = (
    (df["quality_status"] == "WARN").mean() * 100
)

fail_rate = (
    (df["quality_status"] == "FAIL").mean() * 100
)

mean_latency = df["latency_sec"].mean()
median_latency = df["latency_sec"].median()
p95_latency = np.percentile(
    df["latency_sec"],
    95
)

# =========================
# 18. REFERENCE-FREE SCORE
# =========================
#
# This score evaluates system reliability,
# NOT nutritional accuracy.

component_scores = [
    json_validity,
    structured_completeness,
    numeric_validity,
    macro_consistency,
    100 - zero_macro_rate,
    100 - generic_rate,
    100 - remaining_overconfidence
]

reference_free_score = np.mean(
    component_scores
)

# =========================
# 19. SAVE FINAL RESULTS
# =========================

final_columns = [
    "image",
    "food",
    "portion",
    "protein_g",
    "carbs_g",
    "fat_g",
    "fiber_g",
    "calculated_calories",
    "confidence",
    "calibrated_confidence",
    "risk_score",
    "review_required",
    "diagnostic_reason",
    "quality_status",
    "latency_sec"
]

df[final_columns].to_csv(
    OUTPUT_CSV,
    index=False
)

# =========================
# 20. FINAL PRESENTATION
# =========================

print("\n")
print("=" * 72)
print("📊 FINAL REFERENCE-FREE EVALUATION")
print("=" * 72)

print("\nSTRUCTURAL RELIABILITY")
print(f"JSON validity                 : {json_validity:.2f}%")
print(f"Complete structured output    : {structured_completeness:.2f}%")
print(f"Numeric validity              : {numeric_validity:.2f}%")

print("\nNUTRITION PIPELINE CONSISTENCY")
print(f"Macro-calorie consistency     : {macro_consistency:.2f}%")
print(f"Zero-major-macro outputs      : {zero_macro_rate:.2f}%")
print(f"Very-low-calorie outputs      : {low_calorie_rate:.2f}%")

print("\nSEMANTIC & CONFIDENCE RELIABILITY")
print(f"Generic food labels           : {generic_rate:.2f}%")
print(f"Original overconfidence       : {original_overconfidence:.2f}%")
print(f"Remaining high-confidence risk: {remaining_overconfidence:.2f}%")
print(f"Outputs requiring review     : {review_rate:.2f}%")

print("\nQUALITY CLASSIFICATION")
print(f"PASS                          : {pass_rate:.2f}%")
print(f"WARN                          : {warn_rate:.2f}%")
print(f"FAIL                          : {fail_rate:.2f}%")

print("\nLATENCY")
print(f"Mean                          : {mean_latency:.3f} sec")
print(f"Median                        : {median_latency:.3f} sec")
print(f"P95                           : {p95_latency:.3f} sec")

print("\nREFERENCE-FREE SYSTEM SCORE")
print(f"Reliability score             : {reference_free_score:.2f}/100")

print("\n" + "=" * 72)
print("🍽️ NUTRITION ANALYSIS RESULTS")
print("=" * 72)

display(
    df[
        [
            "image",
            "food",
            "portion",
            "protein_g",
            "carbs_g",
            "fat_g",
            "fiber_g",
            "calculated_calories",
            "calibrated_confidence",
            "review_required",
            "quality_status"
        ]
    ].reset_index(drop=True)
)

print("\n" + "=" * 72)
print("⚠️ IMPORTANT EVALUATION NOTE")
print("=" * 72)

print(
    "The evaluation is reference-free. "
    "It measures structural reliability, numerical consistency, "
    "risk detection, confidence calibration, and latency. "
    "It does NOT establish nutritional accuracy because no "
    "ground-truth nutrition references were used."
)

print("\n" + "=" * 72)
print("✓ PROJECT COMPLETE")
print("=" * 72)

print(f"\nResults saved to:")
print(OUTPUT_CSV)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 22.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
🍎 AI NUTRITION COACH
Multimodal AI Nutrition Analysis System
Python        : 3.13.15
PyTorch       : 2.14.0+cu13

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


✓ Model loaded successfully
✓ Model: HuggingFaceTB/SmolVLM2-2.2B-Instruct

------------------------------------------------------------------------
Evaluation Dataset
------------------------------------------------------------------------
Images found : 12

------------------------------------------------------------------------
Running Multimodal Food Analysis
------------------------------------------------------------------------
[01/12] Delicious-Cheeseburger-Close-Up-1536x1024 (1).webp


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

[12/12] amirali-mirhashemian-sc5sTPMrVfk-unsplash (3).jpg


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`




📊 FINAL REFERENCE-FREE EVALUATION

STRUCTURAL RELIABILITY
JSON validity                 : 100.00%
Complete structured output    : 100.00%
Numeric validity              : 100.00%

NUTRITION PIPELINE CONSISTENCY
Macro-calorie consistency     : 100.00%
Zero-major-macro outputs      : 0.00%
Very-low-calorie outputs      : 0.00%

SEMANTIC & CONFIDENCE RELIABILITY
Generic food labels           : 0.00%
Original overconfidence       : 0.00%
Remaining high-confidence risk: 0.00%
Outputs requiring review     : 0.00%

QUALITY CLASSIFICATION
PASS                          : 100.00%
WARN                          : 0.00%
FAIL                          : 0.00%

LATENCY
Mean                          : 6.985 sec
Median                        : 6.934 sec
P95                           : 8.188 sec

REFERENCE-FREE SYSTEM SCORE
Reliability score             : 100.00/100

🍽️ NUTRITION ANALYSIS RESULTS


,image,food,portion,protein_g,carbs_g,fat_g,fiber_g,calculated_calories,calibrated_confidence,review_required,quality_status
0,Delicious-Cheeseburger-Close-Up-1536x1024 (1)....,hamburger,1/4 of a burger,20.0,10.0,10.0,2.0,210.0,high,False,PASS
1,Delicious-Cheeseburger-Close-Up-1536x1024 (2)....,hamburger,1/4 of a burger,20.0,10.0,10.0,2.0,210.0,high,False,PASS
2,Delicious-Cheeseburger-Close-Up-1536x1024 (3)....,hamburger,1/4 of a burger,20.0,10.0,10.0,2.0,210.0,high,False,PASS
3,Delicious-Cheeseburger-Close-Up-1536x1024.webp,hamburger,1/4 of a burger,20.0,10.0,10.0,2.0,210.0,high,False,PASS
4,Single-Food-Salad-1-1152x1536 (1).jpg,salad,1 cup,30.0,20.0,5.0,5.0,245.0,high,False,PASS
5,Single-Food-Salad-1-1152x1536 (2).jpg,salad,1 cup,30.0,20.0,5.0,5.0,245.0,high,False,PASS
6,Single-Food-Salad-1-1152x1536 (3).jpg,salad,1 cup,30.0,20.0,5.0,5.0,245.0,high,False,PASS
7,Single-Food-Salad-1-1152x1536 (4).jpg,salad,1 cup,30.0,20.0,5.0,5.0,245.0,high,False,PASS
8,Single-Food-Salad-1-1152x1536.jpg,salad,1 cup,30.0,20.0,5.0,5.0,245.0,high,False,PASS
9,amirali-mirhashemian-sc5sTPMrVfk-unsplash (1).jpg,hamburger,1/4 of a burger,20.0,10.0,10.0,0.0,210.0,high,False,PASS



⚠️ IMPORTANT EVALUATION NOTE
The evaluation is reference-free. It measures structural reliability, numerical consistency, risk detection, confidence calibration, and latency. It does NOT establish nutritional accuracy because no ground-truth nutrition references were used.

✓ PROJECT COMPLETE

Results saved to:
/content/AI_Nutrition_Final_Validation.csv


In [ ]:
# ============================================================
# AI NUTRITION COACH → GITHUB UPLOADER
# CORRECTED VERSION
# ============================================================

import os
import json
import base64
import getpass
import requests
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

GITHUB_USERNAME = "junaidshah2001"
REPO_NAME = "AI-Nutrition-Coach"
BRANCH = "main"

print("=" * 70)
print("AI NUTRITION COACH — GITHUB UPLOADER")
print("=" * 70)

# ============================================================
# GITHUB TOKEN
# ============================================================

GITHUB_TOKEN = getpass.getpass(
    "Enter your GitHub Personal Access Token: "
).strip()

if not GITHUB_TOKEN:
    raise ValueError("GitHub token was not provided.")

HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

API = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"

# ============================================================
# VERIFY REPOSITORY
# ============================================================

response = requests.get(API, headers=HEADERS)

if response.status_code != 200:
    raise RuntimeError(
        f"Repository access failed.\n"
        f"HTTP Status: {response.status_code}\n"
        f"{response.text[:500]}"
    )

print("✓ GitHub repository verified")
print(f"✓ Repository: {GITHUB_USERNAME}/{REPO_NAME}")

# ============================================================
# CREATE README
# ============================================================

readme_lines = [
    "# AI Nutrition Coach",
    "",
    "A multimodal AI nutrition analysis system that analyzes food images and produces structured nutritional estimates using a vision-language model.",
    "",
    "## Overview",
    "",
    "This project demonstrates an end-to-end multimodal AI pipeline for food image understanding and nutrition estimation.",
    "",
    "The system:",
    "",
    "1. Accepts a food image",
    "2. Uses a vision-language model to identify visible food",
    "3. Estimates the visible portion",
    "4. Estimates protein, carbohydrates, fat, and fiber",
    "5. Calculates calories deterministically from macronutrients",
    "6. Performs structural and consistency validation",
    "7. Detects suspicious outputs",
    "8. Calibrates model confidence",
    "9. Flags uncertain cases for human review",
    "",
    "## Model",
    "",
    "**Hugging Face SmolVLM2-2.2B-Instruct**",
    "",
    "Model: `HuggingFaceTB/SmolVLM2-2.2B-Instruct`",
    "",
    "The model is downloaded automatically at runtime.",
    "",
    "## Architecture",
    "",
    "```text",
    "Food Image",
    "    |",
    "    v",
    "Vision-Language Model",
    "    |",
    "    +-- Food identification",
    "    +-- Portion estimation",
    "    +-- Protein",
    "    +-- Carbohydrates",
    "    +-- Fat",
    "    +-- Fiber",
    "    |",
    "    v",
    "Structured JSON",
    "    |",
    "    v",
    "Deterministic Calorie Calculation",
    "    |",
    "    v",
    "Reference-Free Reliability Audit",
    "    |",
    "    +-- Zero-macro detection",
    "    +-- Generic-label detection",
    "    +-- Very-low-calorie detection",
    "    +-- Macro consistency",
    "    +-- Confidence calibration",
    "    +-- Human-review flag",
    "    |",
    "    v",
    "Final Nutrition Estimate",
    "```",
    "",
    "## Calorie Calculation",
    "",
    "Calories are calculated deterministically using the standard macronutrient energy factors:",
    "",
    "```text",
    "Calories = 4 × Protein(g)",
    "         + 4 × Carbohydrates(g)",
    "         + 9 × Fat(g)",
    "```",
    "",
    "This prevents mathematical inconsistency between reported macronutrients and calories.",
    "",
    "## Validation",
    "",
    "The project includes reference-free validation for:",
    "",
    "- JSON validity",
    "- Structured output completeness",
    "- Numeric validity",
    "- Macro-calorie consistency",
    "- Zero-major-macro detection",
    "- Generic food-label detection",
    "- Suspiciously low calorie detection",
    "- Confidence calibration",
    "- Human-review requirements",
    "",
    "## Important Limitation",
    "",
    "**Reference-free validation is not nutritional accuracy.**",
    "",
    "Actual nutritional accuracy requires a ground-truth dataset containing verified food portions and nutrient values.",
    "",
    "Therefore, this project does not claim that the model's estimated calories or macronutrients are medically or nutritionally exact.",
    "",
    "## Hardware",
    "",
    "The system was developed and tested in Google Colab using:",
    "",
    "- NVIDIA Tesla T4 GPU",
    "- CUDA acceleration",
    "- PyTorch",
    "- Hugging Face Transformers",
    "",
    "## Repository Structure",
    "",
    "```text",
    "AI-Nutrition-Coach/",
    "|",
    "+-- AI_Nutrition_Coach.ipynb",
    "+-- README.md",
    "+-- requirements.txt",
    "+-- results/",
    "    +-- validation_results.csv",
    "```",
    "",
    "## Running the Project",
    "",
    "Open the notebook in Google Colab and run the cells from top to bottom.",
    "",
    "The model will automatically download from Hugging Face.",
    "",
    "## Technologies",
    "",
    "- Python",
    "- PyTorch",
    "- Hugging Face Transformers",
    "- SmolVLM2",
    "- Computer Vision",
    "- Vision-Language Models",
    "- Multimodal AI",
    "- Prompt Engineering",
    "- Structured JSON Generation",
    "- Model Validation",
    "- Confidence Calibration",
    "- Google Colab",
    "",
    "## Project Purpose",
    "",
    "This project demonstrates practical skills in:",
    "",
    "- Multimodal AI",
    "- Vision-Language Models",
    "- Prompt engineering",
    "- Structured model outputs",
    "- Deterministic post-processing",
    "- AI reliability engineering",
    "- Model evaluation",
    "- GPU inference",
    "",
    "## Disclaimer",
    "",
    "This is an experimental AI research and portfolio project.",
    "",
    "The generated nutrition estimates should not be treated as medical, dietary, or clinical advice.",
    ""
]

readme_content = "\n".join(readme_lines)

README_PATH = "/content/README.md"

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✓ README.md created")

# ============================================================
# CREATE REQUIREMENTS
# ============================================================

requirements_content = "\n".join([
    "transformers",
    "accelerate",
    "torch",
    "torchvision",
    "pillow",
    "pandas",
    "numpy",
    "gradio"
]) + "\n"

REQUIREMENTS_PATH = "/content/requirements.txt"

with open(REQUIREMENTS_PATH, "w", encoding="utf-8") as f:
    f.write(requirements_content)

print("✓ requirements.txt created")

# ============================================================
# FIND VALIDATION CSV
# ============================================================

csv_candidates = [
    "/content/AI_Nutrition_Final_Validation.csv",
    "/content/AI_Nutrition_V5_Reliability_Audit.csv",
    "/content/AI_Nutrition_V4_Reference_Free_Audit.csv",
    "/content/AI_Nutrition_V3_Validation.csv"
]

csv_file = None

for candidate in csv_candidates:
    if os.path.exists(candidate):
        csv_file = candidate
        break

if csv_file:
    print(f"✓ Validation CSV found: {os.path.basename(csv_file)}")
else:
    print("⚠ No validation CSV found")

# ============================================================
# CAPTURE CURRENT COLAB NOTEBOOK
# ============================================================

NOTEBOOK_PATH = "/content/AI_Nutrition_Coach.ipynb"

try:
    from google.colab import _message

    notebook_response = _message.blocking_request(
        "get_ipynb",
        timeout_sec=30
    )

    notebook_data = notebook_response

    if isinstance(notebook_data, dict):
        if "ipynb" in notebook_data:
            notebook_data = notebook_data["ipynb"]

    if isinstance(notebook_data, str):
        notebook_data = json.loads(notebook_data)

    if (
        isinstance(notebook_data, dict)
        and "cells" in notebook_data
    ):
        with open(NOTEBOOK_PATH, "w", encoding="utf-8") as f:
            json.dump(
                notebook_data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print("✓ Current Colab notebook captured")

    else:
        print("⚠ Notebook data could not be captured")

except Exception as e:
    print(f"⚠ Notebook capture failed: {e}")

# ============================================================
# PREPARE FILES
# ============================================================

files_to_upload = []

if os.path.exists(NOTEBOOK_PATH):
    files_to_upload.append(
        (NOTEBOOK_PATH, "AI_Nutrition_Coach.ipynb")
    )

files_to_upload.append(
    (README_PATH, "README.md")
)

files_to_upload.append(
    (REQUIREMENTS_PATH, "requirements.txt")
)

if csv_file:
    files_to_upload.append(
        (csv_file, "results/validation_results.csv")
    )

print("\nFiles ready:")
for local_path, remote_path in files_to_upload:
    print(f"  ✓ {remote_path}")

# ============================================================
# GITHUB UPLOAD FUNCTION
# ============================================================

def upload_file(local_path, remote_path):

    with open(local_path, "rb") as f:
        encoded_content = base64.b64encode(
            f.read()
        ).decode("utf-8")

    url = f"{API}/contents/{remote_path}"

    check = requests.get(
        url,
        headers=HEADERS,
        params={"ref": BRANCH}
    )

    payload = {
        "message": f"Add/update {remote_path}",
        "content": encoded_content,
        "branch": BRANCH
    }

    # Existing file → update it
    if check.status_code == 200:
        existing_file = check.json()
        payload["sha"] = existing_file["sha"]

    upload_response = requests.put(
        url,
        headers=HEADERS,
        json=payload
    )

    if upload_response.status_code in [200, 201]:
        print(f"✓ Uploaded: {remote_path}")
        return True

    print(f"✗ Failed: {remote_path}")
    print(upload_response.text[:500])
    return False

# ============================================================
# UPLOAD
# ============================================================

print("\n" + "=" * 70)
print("UPLOADING PROJECT TO GITHUB")
print("=" * 70)

uploaded = 0

for local_path, remote_path in files_to_upload:

    if upload_file(
        local_path,
        remote_path
    ):
        uploaded += 1

# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("GITHUB UPLOAD COMPLETE")
print("=" * 70)

print(
    f"Uploaded: {uploaded}/{len(files_to_upload)} files"
)

print(
    "\nRepository:"
)

print(
    f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}"
)

print("\nExpected structure:")
print("""
AI-Nutrition-Coach/
|
+-- AI_Nutrition_Coach.ipynb
+-- README.md
+-- requirements.txt
|
+-- results/
    +-- validation_results.csv
""")

print("\n✓ Done.")

AI NUTRITION COACH — GITHUB UPLOADER
Enter your GitHub Personal Access Token: ··········
